# Module 6 — Final Ensemble and Error Analysis

Combine the four selected model probabilities with validation-Macro-F1 weights, evaluate one final prediction, and inspect incorrect test examples.

In [ ]:
from pathlib import Path
import sys
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLASS_NAMES, RESULTS_DIR
from src.dataset_utils import load_fixed_data_splits
from src.evaluation import calculate_metrics, save_evaluation_outputs, update_metrics_file
from src.prediction import get_ensemble_predictor

_, validation_data, test_data = load_fixed_data_splits()
predictor = get_ensemble_predictor()

## Evaluate the validation-weighted ensemble

In [ ]:
validation_probabilities = predictor.predict_probabilities(validation_data['original_text'].tolist())
validation_predictions = [CLASS_NAMES[index] for index in validation_probabilities.argmax(axis=1)]
validation_f1 = calculate_metrics(validation_data['label'], validation_predictions)['Macro F1']

test_probabilities = predictor.predict_probabilities(test_data['original_text'].tolist())
test_predictions = [CLASS_NAMES[index] for index in test_probabilities.argmax(axis=1)]
result = save_evaluation_outputs(
    experiment_id='M6.1',
    experiment_name='Validation-Weighted NLP Ensemble',
    family='Combined NLP System',
    test_data=test_data,
    predictions=test_predictions,
)
result['Representation'] = 'TF-IDF + Word2Vec + BiLSTM + Transformer'
result['Validation Macro F1'] = validation_f1
selected_metrics = update_metrics_file(pd.DataFrame([result]))
selected_metrics

## Categorize real ensemble errors

In [ ]:
negation_pattern = re.compile(r'\b(?:not|no|never|na|nai|nei|nay|nahi|dont|doesnt|didnt|isnt|cant|wont)\b')
contrast_pattern = re.compile(r'\b(?:but|however|kintu|tobe)\b')
spelling_variants = {'bhalo', 'valo', 'vhalo', 'vaalo', 'kharap', 'kharappp'}

def categorize_error(row):
    text = str(row['processed_text']).lower()
    tokens = text.split()
    if len(tokens) <= 3:
        return 'Very short text'
    if negation_pattern.search(text):
        return 'Negation'
    if contrast_pattern.search(text) or row['label'] == 'Mixed':
        return 'Mixed or contrastive sentiment'
    if spelling_variants.intersection(tokens):
        return 'Romanized spelling variation'
    return 'Other / manual review needed'

error_analysis = test_data[['sample_id', 'original_text', 'processed_text', 'label']].copy()
error_analysis['predicted_label'] = test_predictions
error_analysis['confidence'] = test_probabilities.max(axis=1)
error_analysis = error_analysis[error_analysis['label'] != error_analysis['predicted_label']].copy()
error_analysis['error_category'] = error_analysis.apply(categorize_error, axis=1)
error_analysis.to_csv(RESULTS_DIR / 'error_analysis.csv', index=False)
display(error_analysis['error_category'].value_counts())
display(error_analysis.head(20))